# Chapter 4 — Figures


**Input**: `ch4_results/` — `results_over_k_all_orbits.csv` and the six
`ci_curves_*.parquet` files.

**Output**: 38 figures whose names match the `\includegraphics` paths in
`Optimalblocksize.tex` and `Appendix.tex`, so the output folder is a drop-in
replacement for `Chapter4/figure/`. All at 300 dpi.

| figures | |
|---|---|
| 6 | `<map>_<obs>_gev_parameters_over_k` — mu, sigma, xi at `i*` against `k` |
| 2 | `<map>_optimal_blocklength_over_k` — `i*(k)`, one panel per observable |
| 30 | `<map>_<obs>_*_vs_i` — appendix diagnostics, orbit 1 |



In [ ]:
!pip install -q numpy pandas scipy matplotlib pyarrow
print("dependencies ready")

In [ ]:
import glob
import json
import os

import matplotlib as mpl
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit

import matplotlib.pyplot as plt

DATA_DIR = "ch4_results"      # from the analysis notebook
OUT_DIR  = "ch4_figures"      # copy into Chapter4/figure/
EXT      = "png"
print("imports ok")

## Configuration



In [ ]:
# =====================================================================
#  CONFIGURATION
# =====================================================================

# --- output ----------------------------------------------------------
SAVE_DPI = 300                

# --- palette ---------------------------------------------------------
WARM_WHITE = "#FBF7F0"         # plot ground; the canvas around it stays white
UQ_NAVY = "#19224C"
BLUE = "#3B4CC0"               # estimates
INK = "#241F1B"                # text, ticks, labels
RULE = "#8A8279"               # legend borders
GRID = "#C9C1B5"

# --- aggregation across trajectories ---------------------------------
STAT = "mean"                  # "mean" or "median";

# --- theory curve ----------------------------------------------------
BLOCK_ADJUST = False           # --block-adjust overrides

# --- i*(k) fit ------------------------------------------------
FIT_FROM_K = 1                 # smallest k in the fit

# --- legends ---------------------------------------------------------
LEGEND_LOC = "upper right"     # body panels; natural size, anchored top right
LEGEND_HEADROOM = 0.50         # blank space reserved above the data
LEGEND_HEADROOM_ISTAR = 0.34
APPENDIX_LEGEND_FONTSIZE = 15  # appendix k-legend, drawn outside the axes


# --- labels ----------------------------------------------------------
PRETTY = {"frechet": "Fréchet", "gumbel": "Gumbel", "weibull": "Weibull"}
XI_STR = {"frechet": r"\xi=0.3", "gumbel": r"\xi=0", "weibull": r"\xi=-0.3"}
OBS_ORDER = ("frechet", "gumbel", "weibull")
MAP_PRETTY = {"doubling": "Doubling", "logistic": "Logistic", "tent": "Tent"}


def set_style():
    """Stock matplotlib defaults; only sizes and dpi are overridden."""
    mpl.rcParams.update(mpl.rcParamsDefault)
    mpl.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 300,
        "font.size": 19, "axes.titlesize": 22, "axes.labelsize": 21,
        "xtick.labelsize": 18, "ytick.labelsize": 18, "legend.fontsize": 16,
        "figure.facecolor": "white",
        "axes.facecolor": WARM_WHITE,
        "savefig.facecolor": "white",
        "axes.edgecolor": "#3A3632",
        "legend.facecolor": "white",
        "legend.edgecolor": "#8A8279",
        "grid.color": "#C9C1B5",
        "text.color": "#241F1B",
        "axes.labelcolor": "#241F1B",
        "xtick.color": "#241F1B",
        "ytick.color": "#241F1B",
    })


# ---------------------------------------------------------------------
# Theoretical scaling under the k-window moving-minimum transformation
#   xi > 0 (Frechet):  mu(k) = lam^{-(k-1)xi} mu_1
#   xi < 0 (Weibull):  mu(k) = mu_1 + (sig_1/xi)(lam^{-(k-1)xi} - 1)
#   xi = 0 (Gumbel):   mu(k) = mu_1 - sig_1 (k-1) log lam
#   sigma(k) = lam^{-(k-1)xi} sig_1   (Frechet & Weibull);  sig_1 (Gumbel)
#   xi(k)    = xi                     
# ---------------------------------------------------------------------


def block_aggregate(mu, sig, xi, j, obs=None):
    """Chapter 4 block-maxima scaling from the reference block to j times it."""
    j = np.asarray(j, float)
    if obs == "gumbel" or abs(xi) < 1e-10:
        return mu + sig * np.log(j), sig
    return mu + sig * ((j ** xi - 1.0) / xi), sig * (j ** xi)

## Theory curves


In [ ]:
def theory_mu(k, lam, xi, mu1, sig1, obs=None):
    k = np.asarray(k, float)
    if obs == "gumbel" or abs(xi) < 1e-10:
        return mu1 - sig1 * (k - 1) * np.log(lam)
    if obs == "frechet" or (obs is None and xi > 0):
        return (lam ** (-(k - 1) * xi)) * mu1
    return mu1 + (sig1 / xi) * (lam ** (-(k - 1) * xi) - 1.0)


def theory_sigma(k, lam, xi, sig1):
    k = np.asarray(k, float)
    if abs(xi) < 1e-10:
        return np.full_like(k, sig1)
    return (lam ** (-(k - 1) * xi)) * sig1


def mu_label(o):
    if o == "gumbel":
        return r"$\mu(k)=\mu_1-\sigma_1(k-1)\log\lambda$"
    if o == "frechet":
        return r"$\mu(k)=\lambda^{-(k-1)\xi}\mu_1$"
    return r"$\mu(k)=\mu_1+\frac{\sigma_1}{\xi}(\lambda^{-(k-1)\xi}-1)$"


def sigma_label(o):
    return (r"$\sigma(k)=\sigma_1$" if o == "gumbel"
            else r"$\sigma(k)=\lambda^{-(k-1)\xi}\sigma_1$")


def values_label(o, lam, xt):
    return (fr"$\lambda={lam:.2f},\ \xi=0$" if o == "gumbel"
            else fr"$\lambda={lam:.2f},\ \xi={xt:.2f}$")


def _central(v, stat=None):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if not v.size:
        return np.nan
    return float(np.mean(v) if (stat or STAT) == "mean" else np.median(v))


def _save(fig, out_path):
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    fig.savefig(out_path, bbox_inches="tight", dpi=SAVE_DPI)
    plt.close(fig)
    print("wrote", out_path)



## Chapter figures

The eight panels in the body of the chapter.

In [ ]:
def plot_params_over_k_single(df, map_name, obs, out_path, show_box=False):
    """3 stacked panels (mu, sigma, xi) at i*, scattered over orbits, vs k.

    Reproduces Chapter4/figure/<map>_<obs>_gev_parameters_over_k.png
    """
    set_style()
    sub = df[(df["map"] == map_name) & (df["observable"] == obs)]
    if sub.empty:
        print("skip (no rows):", map_name, obs)
        return
    k_values = sorted(sub["k"].unique())
    lam = float(sub["lambda"].iloc[0])
    xt = float(sub["xi_true"].iloc[0])
    kline = np.linspace(min(k_values), max(k_values), 200)

    # reference parameters: median across orbits at k = 1
    d1 = sub[sub["k"] == 1]
    mu1 = _central(d1["mu_hat"].to_numpy(float))
    sig1 = _central(d1["sigma_hat"].to_numpy(float))

    fig, axes = plt.subplots(3, 1, figsize=(10.5, 12.5), sharex=True)
    row_ylab = [r"$\widehat{\mu}_{i^{*}}$",
                r"$\widehat{\sigma}_{i^{*}}$",
                r"$\widehat{\xi}_{i^{*}}$"]

    for row, col in enumerate(["mu_hat", "sigma_hat", "xi_hat"]):
        ax = axes[row]
        data = [sub.loc[sub["k"] == k, col].to_numpy(float) for k in k_values]
        data = [d[np.isfinite(d)] for d in data]

        if show_box:
            ax.boxplot(
                data, positions=k_values, widths=0.55, patch_artist=True,
                manage_ticks=False, showfliers=False, zorder=2,
                medianprops=dict(color=UQ_NAVY, linewidth=1.2),
                whiskerprops=dict(color=BLUE, linewidth=0.9),
                capprops=dict(color=BLUE, linewidth=0.9),
                boxprops=dict(facecolor=BLUE, alpha=0.25, edgecolor=BLUE, linewidth=1.0),
            )
        for ki, k in enumerate(k_values):
            v = data[ki]
            if v.size:
                ax.scatter(np.full(v.size, k), v, s=26, color=BLUE,
                           alpha=0.40, edgecolors="none", zorder=3)
        # theory curve, optionally composed with the block-length adjustment
        tm = theory_mu(kline, lam, xt, mu1, sig1, obs=obs)
        ts = theory_sigma(kline, lam, xt, sig1)
        if BLOCK_ADJUST:
            istar_k = np.array([np.nanmean(sub.loc[sub["k"] == kk, "i_star"]
                                           .to_numpy(float)) for kk in k_values])
            i1 = istar_k[0]
            if np.isfinite(i1) and i1 > 0:
                jline = np.interp(kline, k_values, istar_k) / i1
                tm, ts = block_aggregate(tm, ts, xt, jline, obs=obs)

        if row == 0:
            ax.plot(kline, tm, color="red", lw=2.0, zorder=5)
            eq, val = mu_label(obs), values_label(obs, lam, xt)
        elif row == 1:
            ax.plot(kline, ts, color="red", lw=2.0, zorder=5)
            eq, val = sigma_label(obs), values_label(obs, lam, xt)
        else:
            ax.axhline(xt, color="red", lw=2.0, zorder=5)
            eq = fr"$\xi(k)=\xi_{{\mathrm{{true}}}}={xt:.2f}$"
            val = None

        h = [Line2D([0], [0], marker="o", color="none", markerfacecolor=BLUE,
                    alpha=0.6, markersize=6, label="Estimated values"),
             Line2D([0], [0], color="red", lw=2.0, label=eq)]
        if val is not None:
            h.append(Line2D([0], [0], color="none", label=val))

        ax.set_xticks(k_values)
        ax.set_xlim(min(k_values) - 0.5, max(k_values) + 0.5)
        ax.set_ylabel(row_ylab[row])
        ax.grid(True, ls=":", lw=0.6, alpha=0.6)

        
        allv = np.concatenate([v for v in data if v.size]) if any(
            v.size for v in data) else np.array([0.0])
        d_lo, d_hi = float(allv.min()), float(allv.max())
        cur = np.asarray(tm if row == 0 else (ts if row == 1 else [xt]), float)
        cur = cur[np.isfinite(cur)]
        if cur.size:
            d_lo, d_hi = min(d_lo, float(cur.min())), max(d_hi, float(cur.max()))
        if row == 2:
            d_lo, d_hi = min(d_lo, xt), max(d_hi, xt)
        pad = 0.05 * (d_hi - d_lo) if d_hi > d_lo else 1.0
        ax.set_ylim(d_lo - pad, d_hi + pad)

        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi + LEGEND_HEADROOM * (hi - lo))
        
        ax.legend(handles=h, loc=LEGEND_LOC, frameon=True, facecolor="white",
                  framealpha=1.0, edgecolor="#8A8279", fancybox=False,
                  borderpad=0.5, handlelength=1.6).set_zorder(10)

    axes[-1].set_xlabel(r"Window size $k$")
    fig.suptitle(f"{MAP_PRETTY.get(map_name, map_name)} map — {PRETTY[obs]}",
                 fontsize=26, y=0.995)
    fig.tight_layout()
    _save(fig, out_path)


def _istar_model(k, A, B, lam, xi):
    k = np.asarray(k, float)
    if abs(xi) < 1e-10:                       
        return A * (k - 1) + B
    return A * (lam ** ((k - 1) * abs(xi))) + B


def _fit_istar(kv, med, lam, xi, fit_from=None):
    kv = np.asarray(kv, float)
    med = np.asarray(med, float)
    fit_from = FIT_FROM_K if fit_from is None else fit_from
    ok = np.isfinite(med) & (kv >= fit_from)
    if ok.sum() < 3:                      # fall back to the whole range
        ok = np.isfinite(med)
    if ok.sum() < 3:
        return None
    f = lambda k, A, B: _istar_model(k, A, B, lam, xi)  # noqa: E731
    try:
        popt, _ = curve_fit(f, kv[ok], med[ok], p0=[med[ok][0], 0.0], maxfev=8000)
    except Exception:
        return None
    pred = f(kv[ok], *popt)
    resid = med[ok] - pred
    ssr = float(np.sum(resid ** 2))
    sst = float(np.sum((med[ok] - med[ok].mean()) ** 2))
    r2 = 1 - ssr / sst if sst > 0 else np.nan
    rmse = float(np.sqrt(ssr / ok.sum()))
    return popt, r2, rmse, float(kv[ok].min())


def plot_istar_over_k(df, map_name, out_path, show_box=False):
    """
    Reproduces Chapter4/figure/<map>_optimal_blocklength_over_k.png
    """
    set_style()
    dm = df[df["map"] == map_name]
    if dm.empty:
        print("skip (no rows):", map_name)
        return
    k_values = sorted(dm["k"].unique())
    lam = float(dm["lambda"].iloc[0])

    fig, axes = plt.subplots(len(OBS_ORDER), 1,
                             figsize=(10.5, 12.5), sharex=True)
    axes = np.atleast_1d(axes)

    for ax, obs in zip(axes, OBS_ORDER):
        sub = dm[dm["observable"] == obs]
        xt = float(sub["xi_true"].iloc[0])
        data = [sub.loc[sub["k"] == k, "i_star"].to_numpy(float) for k in k_values]
        data = [d[np.isfinite(d)] for d in data]

        if show_box:
            ax.boxplot(
                data, positions=k_values, widths=0.55, patch_artist=True,
                manage_ticks=False, showfliers=False, zorder=2,
                medianprops=dict(color=UQ_NAVY, linewidth=1.2),
                whiskerprops=dict(color=BLUE, linewidth=0.9),
                capprops=dict(color=BLUE, linewidth=0.9),
                boxprops=dict(facecolor=BLUE, alpha=0.25, edgecolor=BLUE, linewidth=1.0),
            )
        for ki, k in enumerate(k_values):
            v = data[ki]
            if v.size:
                ax.scatter(np.full(v.size, k), v, s=30, color="0.25",
                           alpha=0.45, edgecolors="none", zorder=3)

        med = np.array([_central(d) for d in data])
        fit = _fit_istar(k_values, med, lam, xt)

        if fit is not None:
            (A, B), r2, rmse, k0 = fit
            kline = np.linspace(min(k_values), max(k_values), 200)
            ax.plot(kline, _istar_model(kline, A, B, lam, xt),
                    color="red", ls=":", lw=2.6, zorder=5)
            if abs(xt) < 1e-10:
                eq = fr"$y={B:.2f}+{A:.2f}\,(k-1)$"
            else:
                sgn = "+" if B >= 0 else "-"
                eq = (fr"$y={A:.2f}\,\lambda^{{(k-1)|\xi|}}{sgn}{abs(B):.2f}$")
            h = [Line2D([0], [0], marker="o", color="none", markerfacecolor="0.25",
                        alpha=0.6, markersize=6, label="Block length"),
                 Line2D([0], [0], color="red", ls=":", lw=2.6, label=eq)]
            # fit quality kept out of the legend box to keep it small
            ax.text(0.985, 0.03, fr"RMSE$={rmse:.0f}$,  $R^2={r2:.2f}$",
                    transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=15, color="0.35")
        else:
            h = [Line2D([0], [0], marker="o", color="none", markerfacecolor="0.25",
                        alpha=0.6, markersize=6, label="Block length")]

        ax.set_ylabel(fr"$i^{{\star}}$ ({PRETTY[obs]})")
        ax.set_xticks(k_values)
        ax.set_xlim(min(k_values) - 0.5, max(k_values) + 0.5)
        ax.grid(True, ls=":", lw=0.6, alpha=0.6)

        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi + LEGEND_HEADROOM_ISTAR * (hi - lo))
        ax.legend(handles=h, loc=LEGEND_LOC, frameon=True, facecolor="white",
                  framealpha=1.0, edgecolor="#8A8279", fancybox=False,
                  borderpad=0.5, handlelength=1.6).set_zorder(10)

    axes[-1].set_xlabel(r"Window size $k$")
    fig.suptitle(f"{MAP_PRETTY.get(map_name, map_name)} map", fontsize=26, y=0.997)
    fig.tight_layout()
    _save(fig, out_path)



## Appendix figures

The thirty CI-versus-block-length diagnostics, computed from one representative orbit.

In [ ]:
def _k_colors(K):
    cmap = plt.get_cmap("tab10")
    return [cmap(i % 10) for i in range(K)]


def _load(path):
    return pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)


def _ci_panel(ax, dfc, val, lo, hi, colors, kvals, ylabel,
              dashed_at=None, mark_istar=True):
    for idx, k in enumerate(kvals):
        c = dfc[dfc["k"] == k].sort_values("i")
        if c.empty:
            continue
        x = c["i"].to_numpy(float)
        ax.plot(x, c[val], color=colors[idx], lw=1.1, zorder=3)
        ax.fill_between(x, c[lo], c[hi], color=colors[idx],
                        alpha=0.15, linewidth=0, zorder=1)
        if mark_istar:
            istar = c["i_star"].iloc[0]
            if pd.notna(istar) and (c["i"] == istar).any():
                yv = float(c.loc[c["i"] == istar, val].iloc[0])
                ax.plot([istar], [yv], "o", color=colors[idx], ms=7,
                        markeredgecolor="black", markeredgewidth=0.8, zorder=6)
    if dashed_at is not None:
        ax.axhline(dashed_at, ls="--", color="black", lw=1.2, zorder=4)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(r"block size $i$")
    ax.margins(y=0.06)



def _k_legend(ax, colors, kvals, inside=True):
    h = [Line2D([0], [0], color=colors[i], lw=2.4, label=fr"$k={k}$")
         for i, k in enumerate(kvals)]
    common = dict(handles=h, ncol=2, loc="upper left",
                  fontsize=APPENDIX_LEGEND_FONTSIZE, frameon=True,
                  facecolor="white", framealpha=1.0, edgecolor="#8A8279",
                  fancybox=False, columnspacing=1.2, handlelength=1.6,
                  borderpad=0.6)
    if inside:
        leg = ax.legend(labelspacing=0.4, **common)
    else:
        leg = ax.legend(bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0,
                        labelspacing=0.5, **common)
    leg.set_zorder(10)
    return leg


# five single-panel appendix figures per (map, observable)
_APPENDIX_PANELS = [
    # suffix,          value,     lo,           hi,           ylabel,                                   dashed
    ("raw_loc_vs_i",   "mu",      "mu_lo",      "mu_hi",      r"$\widehat{\mu}$ (loc)",                 False),
    ("raw_scale_vs_i", "sigma",   "sigma_lo",   "sigma_hi",   r"$\widehat{\sigma}$ (scale)",            False),
    ("mu_star_vs_i",   "mu_s",    "mu_s_lo",    "mu_s_hi",    r"$\widehat{\mu}^{*}$ (rescaled loc)",    False),
    ("sigma_star_vs_i", "sigma_s", "sigma_s_lo", "sigma_s_hi", r"$\widehat{\sigma}^{*}$ (rescaled scale)", False),
    ("xi_vs_i",        "xi",      "xi_lo",      "xi_hi",      r"$\widehat{\xi}$ (shape)",               True),
]


def plot_appendix_singles(dfc, map_name, obs, out_dir, ext, orbit_label,
                          alpha, mark_istar=True):
    """The five wide single-panel appendix figures for one (map, observable)."""
    set_style()
    kvals = sorted(dfc["k"].unique())
    colors = _k_colors(len(kvals))
    xt = float(dfc["xi_true"].iloc[0])
    lam = float(dfc["lambda"].iloc[0])
    title = (f"{map_name} | {obs}_xi_{xt:g} | orbit={orbit_label} | "
             fr"$\alpha={alpha:g}$ | $\lambda={lam:g}$")

    for suffix, val, lo, hi, ylab, dashed in _APPENDIX_PANELS:
        fig, ax = plt.subplots(1, 1, figsize=(15, 4.8))
        _ci_panel(ax, dfc, val, lo, hi, colors, kvals, ylab,
                  dashed_at=(xt if dashed else None), mark_istar=mark_istar)
        ax.set_title(title, fontsize=19)
        # legend outside, to the right of the axes
        _k_legend(ax, colors, kvals, inside=False)
        fig.tight_layout()
        _save(fig, os.path.join(out_dir, f"{map_name}_{obs}_{suffix}.{ext}"))


def plot_appendix_combined(dfc, map_name, obs, out_dir, ext):
    set_style()
    kvals = sorted(dfc["k"].unique())
    colors = _k_colors(len(kvals))
    xt = float(dfc["xi_true"].iloc[0])

    fig, axes = plt.subplots(2, 1, figsize=(9.6, 8), sharex=True)
    _ci_panel(axes[0], dfc, "mu", "mu_lo", "mu_hi", colors, kvals, r"$\widehat{\mu}$ (location)")
    axes[0].set_title("(a) Non-rescaled location", fontweight="bold", loc="left")
    _ci_panel(axes[1], dfc, "sigma", "sigma_lo", "sigma_hi", colors, kvals, r"$\widehat{\sigma}$ (scale)")
    axes[1].set_title("(b) Non-rescaled scale", fontweight="bold", loc="left")
    _k_legend(axes[0], colors, kvals, inside=False)
    fig.tight_layout()
    _save(fig, os.path.join(out_dir, f"{map_name}_{obs}_appendix_ci_raw_params.{ext}"))

    fig, axes = plt.subplots(2, 1, figsize=(9.6, 8), sharex=True)
    _ci_panel(axes[0], dfc, "mu_s", "mu_s_lo", "mu_s_hi", colors, kvals, r"$\widehat{\mu}^*$ (rescaled location)")
    axes[0].set_title("(d) Rescaled location", fontweight="bold", loc="left")
    _ci_panel(axes[1], dfc, "sigma_s", "sigma_s_lo", "sigma_s_hi", colors, kvals, r"$\widehat{\sigma}^*$ (rescaled scale)")
    axes[1].set_title("(e) Rescaled scale", fontweight="bold", loc="left")
    _k_legend(axes[0], colors, kvals, inside=False)
    fig.tight_layout()
    _save(fig, os.path.join(out_dir, f"{map_name}_{obs}_appendix_ci_rescaled_params.{ext}"))

    fig, ax = plt.subplots(1, 1, figsize=(9.6, 4.5))
    _ci_panel(ax, dfc, "xi", "xi_lo", "xi_hi", colors, kvals, r"$\widehat{\xi}$ (shape)", dashed_at=xt)
    ax.set_title("(c) Shape", fontweight="bold", loc="left")
    _k_legend(ax, colors, kvals, inside=False)
    fig.tight_layout()
    _save(fig, os.path.join(out_dir, f"{map_name}_{obs}_appendix_ci_shape_param.{ext}"))



## Make every figure

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

alpha = 0.05
mpath = os.path.join(DATA_DIR, "manifest.json")
if os.path.exists(mpath):
    alpha = float(json.load(open(mpath)).get("alpha", 0.05))

res = pd.read_csv(os.path.join(DATA_DIR, "results_over_k_all_orbits.csv"))
print(f"{len(res)} rows, {res['orbit_id'].nunique()} orbits, "
      f"k={res['k'].min()}..{res['k'].max()}, stat={STAT}")

# --- chapter figures -------------------------------------------------
for m in sorted(res["map"].unique()):
    for obs in OBS_ORDER:
        plot_params_over_k_single(
            res, m, obs, os.path.join(OUT_DIR, f"{m}_{obs}_gev_parameters_over_k.{EXT}"))
    plot_istar_over_k(res, m, os.path.join(OUT_DIR, f"{m}_optimal_blocklength_over_k.{EXT}"))

# --- appendix figures ------------------------------------------------
for path in sorted(glob.glob(os.path.join(DATA_DIR, "ci_curves_*.parquet")) +
                   glob.glob(os.path.join(DATA_DIR, "ci_curves_*.csv"))):
    core = os.path.basename(path).replace("ci_curves_", "").rsplit(".", 1)[0]
    mn, on, *rest = core.split("_")
    orbit = rest[0].replace("orb", "") if rest else "1"
    dfc = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
    plot_appendix_singles(dfc, mn, on, OUT_DIR, EXT, orbit, alpha)

n = len(glob.glob(os.path.join(OUT_DIR, f"*.{EXT}")))
print(f"\ndone: {n} figures in {OUT_DIR}/")

## Check and save

In [ ]:
from PIL import Image

dpis = {round(Image.open(p).info["dpi"][0]) for p in glob.glob(f"{OUT_DIR}/*.png")}
print("dpi across all figures:", sorted(dpis))

import shutil
shutil.make_archive("ch4_figures", "zip", ".", OUT_DIR)
print("wrote ch4_figures.zip")
try:
    from google.colab import files
    files.download("ch4_figures.zip")
except Exception:
    pass